# Semantic Search System with Fuzzy Clustering and Semantic Cache

This notebook implements a lightweight semantic search system using
the 20 Newsgroups dataset.

Core components:

1. Vector embeddings
2. FAISS vector search
3. Fuzzy clustering using Gaussian Mixture Models
4. Semantic cache built from scratch
5. FastAPI service layer

In [1]:
!pip install sentence-transformers faiss-cpu scikit-learn matplotlib numpy pandas

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.7 MB 1.8 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/10.7 MB 1.8 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/10.7 MB 1.4 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/10.7 MB 1.4 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/10.7 MB 1.1 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/10.7 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.1/10.7 MB 1.2 MB/s eta 0:00:07
   -------- ------------------------------- 2.4/10.7 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/10.7 MB 1.2 MB/s eta 0:00:07
   ---------- ---


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\itsay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import faiss

In [5]:
# Path to the raw 20 Newsgroups dataset.
# The dataset is structured as directories where each folder
# represents a newsgroup category and each file is a single post.

DATASET_PATH = "../data/20_newsgroups"

docs = []
labels = []
categories = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if os.path.isdir(category_path):

        categories.append(category)

        for file in os.listdir(category_path):

            file_path = os.path.join(category_path, file)

            try:
                with open(file_path, "r", encoding="latin1") as f:
                    docs.append(f.read())
                    labels.append(category)
            except:
                continue

print("Total documents:", len(docs))
print("Total categories:", len(categories))
print("Categories:", categories[:10])

Total documents: 19997
Total categories: 20
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball']


## Observations from Raw Documents

The Usenet articles contain extensive metadata headers including:

- Xref
- Path
- From
- Newsgroups
- Subject
- Message-ID
- Organization
- Date

These fields describe message routing and email metadata rather
than the semantic content of the discussion.

Including these tokens would introduce noise into the embedding
space and may cause the model to learn artifacts rather than
true topic relationships.

Therefore the preprocessing step removes headers and retains
only the message body before generating embeddings.

In [6]:
def remove_headers(text):

    # Usenet posts begin with a metadata header block
    # containing routing information such as:
    # From, Subject, Organization, Message-ID, etc.
    #
    # These fields do not represent the semantic topic
    # of the message and can introduce noise into
    # embedding generation.
    #
    # The message body starts after the first blank line.
    
    parts = text.split("\n\n", 1)

    if len(parts) > 1:
        return parts[1]

    return text


# Apply header removal to all documents
clean_docs = [remove_headers(d) for d in docs]